# Cell type annotation: test

Runs the steps of `scripts/seg_postprocessing/cell_type_annotation.py` in memory, with the same
functions and defaults, for any sample and segmentation method (set them in the first cell).
Reuses the existing MapMyCells JSON and writes only to `cell_type_annotation/_scalpel_check/`.

1. MapMyCells output
2. SCALPEL QC per supertype (DoubleMAD, bimodal handling): failed cells -> `Undefined`
3. `group_cell_types()`
4. Leiden majority vote, QC-failed cells voting `Undefined` -> `cell_type_vote`
5. Marker revision with curated markers (reassign only, never `Undefined`) -> `cell_type_revised`

In [ ]:
# ruff: noqa
import os, sys, json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sample_name = "htra1_s3_r0"
seg_method = "Proseg_3D_Cellpose_1_nuclei_model"
data_dir = "/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark"

# same defaults as cell_type_annotation.py
mad_factor, leiden_res, marker_min_score, marker_delta = 3.0, 20.0, 1.0, 0.25

REPO = str(Path.cwd().parents[1])   # this notebook lives in REPO/notebooks/script_setup
assert Path(REPO, "cellseg_benchmark").is_dir(), f"{REPO} is not the repo root"
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith("cellseg_benchmark")]:
    del sys.modules[m]
import cellseg_benchmark.cell_annotation_utils as anno_utils
from cellseg_benchmark._constants import cell_type_colors
print(anno_utils.__file__)
assert hasattr(anno_utils, "annotate_clusters"), f"{anno_utils.__file__} is outdated: pull scalpel-annotation in {REPO}"

warnings.filterwarnings("ignore")
logger = logging.getLogger("annotation_check")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
pd.set_option("display.width", 160)

method_path = Path(data_dir, "samples", sample_name, "results", seg_method)
annotation_path = method_path / "cell_type_annotation"
out_path = annotation_path / "_scalpel_check"
out_path.mkdir(parents=True, exist_ok=True)
marker_csv = Path(data_dir, "misc", "scRNAseq_ref_ABCAtlas_Yao2023Nature", "marker_genes_df",
                  "20250416_cell_type_markers_top50.csv")

## 1. SCALPEL QC

In [ ]:
json_path = max(annotation_path.glob(f"mapmycells_out/*MapMyCells_{sample_name}_{seg_method}.json"),
                key=os.path.getmtime)
with open(json_path, "rb") as src:
    mmc = anno_utils.process_mapmycells_output(json.load(src))
mmc = mmc.join(anno_utils.scalpel_qc(mmc["allen_cor_SUPT"], mmc["allen_SUPT"], mad_factor))

n_per_st = mmc.groupby("allen_SUPT").size()
print(json_path.name)
print(f"cells: {len(mmc):,}   supertypes: {len(n_per_st)}   median cells/supertype: {n_per_st.median():.0f}")
print(f"cells in supertypes with < 10 cells: {n_per_st[n_per_st < 10].sum() / len(mmc):.1%}")
print(f"bimodal supertypes: {mmc.loc[mmc.is_bimodal_supertype, 'allen_SUPT'].nunique()} "
      f"({mmc.is_bimodal_supertype.mean():.1%} of cells)")
print(f"QC failed: {(~mmc.qc_passed).mean():.2%} of cells")

In [ ]:
bimodal = mmc.loc[mmc.is_bimodal_supertype, "allen_SUPT"].value_counts().index[:8].tolist()
largest = mmc.loc[~mmc.is_bimodal_supertype, "allen_SUPT"].value_counts().index[:4].tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, st in zip(axes.flat, bimodal + largest):
    sub = mmc[mmc.allen_SUPT == st]
    ax.hist(sub.allen_cor_SUPT, bins=50, color="C1" if st in bimodal else "C0")
    ax.axvline(sub.qc_thr.max(), color="k", ls="--")
    ax.set_title(f"{st[:35]}\nn={len(sub)}, failed {(~sub.qc_passed).mean():.0%}", fontsize=8)
for ax in axes.flat[len(bimodal + largest):]:
    ax.axis("off")
plt.suptitle("orange: bimodal supertypes, blue: largest unimodal; dashed: threshold")
plt.tight_layout()
plt.savefig(out_path / "1_qc_thresholds.png", dpi=120)
plt.show()

## 2. Cluster vote and marker revision, as in the script

In [ ]:
from spatialdata import read_zarr

mmc["allen_SUBC"] = anno_utils.group_cell_types(mmc["allen_SUBC"]).fillna("Undefined")
mmc["allen_SUBC_incl_low_quality"] = mmc["allen_SUBC"].where(mmc.qc_passed, "Undefined")

adata = read_zarr(method_path / "sdata.zarr")["table"]
adata = adata[:, ~adata.var_names.str.startswith("Blank")]
adata.obsm["allen_cell_type_mapping"] = mmc.loc[adata.obs.index]
adata = anno_utils.process_adata(adata=adata, seg_method=seg_method, logger=logger)

leiden_col = f"leiden_res{leiden_res}".replace(".", "_")
if leiden_col not in adata.obs:
    sc.tl.leiden(adata, key_added=leiden_col, resolution=leiden_res)
adata.obs["cell_type_vote"], adata.obs["cell_type_revised"] = anno_utils.annotate_clusters(
    adata, cluster_col=leiden_col, label_col="cell_type_mmc_incl_low_quality", marker_csv=marker_csv,
    min_score=marker_min_score, delta=marker_delta, logger=logger,
)
print(adata.obs[leiden_col].nunique(), "clusters,", f"{adata.n_obs:,} cells")

In [ ]:
# every marker relabelling, with the winning type's cluster-mean score
cl = adata.obs[leiden_col].astype(str)
scores = adata.obs.filter(like="score_").groupby(cl).mean()
per_cluster = pd.DataFrame({"vote": adata.obs.cell_type_vote.groupby(cl).first(),
                            "revised": adata.obs.cell_type_revised.groupby(cl).first(),
                            "cells": cl.value_counts()})
rel = per_cluster[per_cluster.vote != per_cluster.revised].copy()
rel["score"] = [scores.loc[c, f"score_{t}"] for c, t in rel.revised.items()]
rel.groupby(["vote", "revised"]).agg(clusters=("cells", "size"), cells=("cells", "sum"),
                                     median_score=("score", "median")).round(2)

## 3. Labels at each step

In [ ]:
steps = {
    "1. MMC per cell": adata.obs["cell_type_mmc_raw"].astype(str),
    "2. after SCALPEL QC": adata.obs["cell_type_mmc_incl_low_quality"].astype(str),
    "3. cluster vote": adata.obs["cell_type_vote"],
    "4. cell_type_revised": adata.obs["cell_type_revised"],
}
saved_csv = annotation_path / "adata_obs_annotated.csv"
if saved_csv.exists() and "cell_id" in adata.obs:
    saved = pd.read_csv(saved_csv, usecols=["cell_id", "cell_type_revised"])
    saved = saved.set_index(saved.cell_id.astype(str)).cell_type_revised
    steps["saved cell_type_revised (csv)"] = (adata.obs.cell_id.astype(str).map(saved)
                                              .fillna("not in csv").set_axis(adata.obs.index))

present = set().union(*[set(v.unique()) for v in steps.values()])
cats = [c for c in cell_type_colors if c in present] + sorted(present - set(cell_type_colors))
palette = {c: cell_type_colors.get(c, "#555555") for c in cats}

bases = ["umap"] + (["spatial"] if "spatial" in adata.obsm else [])
fig, axes = plt.subplots(len(steps), len(bases), figsize=(8 * len(bases), 6.5 * len(steps)), squeeze=False)
for r, (title, lab) in enumerate(steps.items()):
    adata.obs["_step"] = pd.Categorical(lab, categories=cats)
    for c, basis in enumerate(bases):
        sc.pl.embedding(adata, basis=basis, color="_step", palette=palette,
                        size=(220000 if basis == "umap" else 110000) / adata.n_obs,
                        legend_loc="on data" if basis == "umap" else None,
                        legend_fontsize=7, legend_fontoutline=1.5,
                        title=f"{title} ({basis})", ax=axes[r, c], show=False)
        axes[r, c].set_aspect("equal")
adata.obs.drop(columns="_step", inplace=True)
fig.legend([plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=palette[k], markersize=8) for k in cats],
           cats, loc="center right", fontsize=9, frameon=False)
plt.subplots_adjust(right=0.85, wspace=0.05, hspace=0.12)
plt.savefig(out_path / "3_labels_by_step.png", dpi=110, bbox_inches="tight")
plt.show()

counts = pd.DataFrame({k: v.value_counts() for k, v in steps.items()}).fillna(0).astype(int)
counts.sort_values(counts.columns[0], ascending=False)

## 4. Vascular markers

In [ ]:
markers = {"ECs": ["Cldn5", "Flt1", "Pecam1"], "Pericytes": ["Pdgfrb", "Kcnj8", "Vtn", "Rgs5"],
           "SMCs": ["Acta2", "Myh11"], "VLMCs": ["Dcn", "Col1a1"]}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}
print("in panel:", markers)

if markers:
    sc.pl.umap(adata, color=sum(markers.values(), []), layer="volume_log1p_norm", ncols=4, cmap="Reds",
               size=220000 / adata.n_obs, vmax="p99", show=False)
    plt.savefig(out_path / "4_vascular_markers_umap.png", dpi=110, bbox_inches="tight")
    plt.show()

    for tag, name in [("revised", "4. cell_type_revised"), ("saved", "saved cell_type_revised (csv)")]:
        if name in steps:
            adata.obs["_grp"] = pd.Categorical(steps[name])
            sc.pl.dotplot(adata, markers, groupby="_grp", layer="volume_log1p_norm",
                          standard_scale="var", title=name, show=False)
            plt.savefig(out_path / f"4_vascular_markers_dotplot_{tag}.png", dpi=110, bbox_inches="tight")
            plt.show()
    adata.obs.drop(columns="_grp", inplace=True, errors="ignore")

    purity = {}
    for name, v in steps.items():
        if name.startswith(("1.", "2.")):
            continue
        row = {"Undefined %": 100 * (v == "Undefined").mean()}
        for t, genes in markers.items():
            expr = sc.get.obs_df(adata, keys=genes, layer="volume_log1p_norm").mean(axis=1)
            row[f"{t} n"], row[f"{t} marker mean"] = (v == t).sum(), expr[v == t].mean()
        purity[name] = row
    display(pd.DataFrame(purity).T.round(2))

## 5. Annotation QC (as exported by the script to `annotation_qc.csv`)

`undefined_mixed_excess`: MapMyCells mixed rate of Undefined minus assigned cells at matched counts,
in percentage points. ~0: Undefined cells are low quality; > 0: mixed identity beyond their counts.

In [ ]:
qc_summary = anno_utils.annotation_qc_summary(adata)
qc_summary.to_csv(out_path / "5_annotation_qc.csv")
counts.to_csv(out_path / "3_counts_by_step.csv")
qc_summary